## 作业1 使用langchain_agent实现pdf和TAVILY工具组合动态调用的实现。

使用西游记前三十回的文本

In [13]:
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_classic import hub
from langchain_classic.tools.retriever import create_retriever_tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader


load_dotenv()

True

In [19]:
model = ChatOpenAI(
    model="qwen/qwen3-235b-a22b-2507",
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url=os.environ['OPENROUTER_BASE_URL']
) # 支持function call

In [20]:
# 提示词模版
prompt = hub.pull("hwchase17/openai-functions-agent")

# 构建工具
search = TavilySearchResults(max_results=2)

embedding_model = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url=os.environ['OPENROUTER_BASE_URL']
)

e:\miniconda3\envs\py3127\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [21]:
if not os.path.exists('local_save'):
    # 加载西游记中文本内容，转换langchain处理的Document
    loader = TextLoader('./xyj.txt', encoding='utf-8')
    docs = loader.load()

    # TextSplitter实现加载后Document分割
    splitter = RecursiveCharacterTextSplitter(
        separators=['\n\n','\n',''],
        chunk_size=1000,
        chunk_overlap=100
    )
    splited_docs = splitter.split_documents(docs)
    
    # 创建向量数据库（内存中）对chunk进行向量化和存储
    vector_store = FAISS.from_documents(
        documents=splited_docs,
        embedding=embedding_model)
    # 向量数据库本地化存储
    vector_store.save_local('local_save')
    print('faiss数据库本地化保存成功！')
else:
    vector_store = FAISS.load_local(
        'local_save', 
        embeddings=embedding_model, 
        allow_dangerous_deserialization=True
    )
    print('加载faiss数据库本地化记录成功！')



faiss数据库本地化保存成功！


In [23]:
vector_store = FAISS.load_local(
        'local_save', 
        embeddings=embedding_model, 
        allow_dangerous_deserialization=True
)
# 构建检索器
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retriever_tool = create_retriever_tool(
    retriever, "book_retriever",
    description="这是西游记前三十回的检索器"
)

# 工具列表
tools = [search,retriever_tool]

# 构建agent
agent = create_tool_calling_agent(
    llm=model, prompt=prompt, tools=tools)

# agent executor
executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True)

# 运行agent 
# msgs = executor.invoke({"input":"北京明天天气如何"})
msgs = executor.invoke({"input":"脱难江流来国土是那一回? "})

print(msgs['output'])



> Entering new AgentExecutor chain...

Invoking: `book_retriever` with `{'query': '脱难江流来国土'}`


沙僧道：“哥啊，定不得吉凶哩。我们且去看来。”二人雄纠纠的到了门前，呀！闭着门哩。只见那门上横安了一块白玉石板，上镌着六个大字：“碗子山波月洞”。沙僧道：“哥啊，这不是甚么寺院，是一座妖精洞府也。我师父在这里，也见不得哩。”八戒道：“兄弟莫怕，你且拴下马匹，守着行李，待我问他的信看。”那呆子举着钯，上前高叫：“开门！开门！”那洞内有把门的小妖开了门，忽见他两个的模样，急抽身跑入里面报道：“大王！买卖来了！”老妖道：“那里买卖？”小妖道：“洞门外有一个长嘴大耳的和尚，与一个晦气色的和尚，来叫门了！”老妖大喜道：“是猪八戒与沙僧寻将来也！噫，他也会寻哩！怎么就寻到我这门上？既然嘴脸凶顽，却莫要怠慢了他。”叫：“取披挂来！”

    小妖抬来，就结束了，绰刀在手，径出门来。

    却说那八戒、沙僧在门前正等，只见妖魔来得凶险。你道他怎生打扮：青脸红须赤发飘，黄金铠甲亮光饶。裹肚衬腰磲石带，攀胸勒甲步云绦。闲立山前风吼吼，闷游海外浪滔滔。一双蓝靛焦筋手，执定追魂取命刀。要知此物名和姓，声扬二字唤黄袍。那黄袍老怪出得门来，便问：“你是那方和尚，在我门首吆喝？”八戒道：“我儿子，你不认得？我是你老爷！我是大唐差往西天去的！我师父是那御弟三藏。若在你家里，趁早送出来，省了我钉钯筑进去！”那怪笑道：“是，是，是有一个唐僧在我家。我也不曾怠慢他，安排些人肉包儿与他吃哩。你们也进去吃一个儿，何如？”这呆子认真就要进去，沙僧一把扯住道：

    “哥啊，他哄你哩，你几时又吃人肉哩？”呆子却才省悟，掣钉钯，望妖怪劈脸就筑。那怪物侧身躲过，使钢刀急架相迎。两个都显神通，纵云头，跳在空中厮杀。沙僧撇了行李白马，举宝杖，急急帮攻。此时两个狠和尚，一个泼妖魔，在云端里，这一场好杀，正是那：杖起刀迎，钯来刀架。一员魔将施威，两个神僧显化。九齿钯真个英雄，降妖伐诚然凶咤。没前后左右齐来，那黄袍公然不怕。你看他蘸钢刀晃亮如银，其实的那神通也为广大。只杀得满空中雾绕云迷、半山里崖崩岭咋。一个为声名，怎肯干休？一个为师父，断然不怕。他三个在半空中，往往来来，战经数十回合，不分胜负。各因性命要紧，其实难解难分

In [24]:
msgs = executor.invoke({"input":"武汉明天天气如何"})
print(msgs['output'])




> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '武汉明天天气'}`


[{'title': '武汉 - 中国气象局-天气预报-城市预报', 'url': 'https://weather.cma.cn/web/weather/57494.html', 'content': '|  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 11.3℃ | 19.8℃ | 22.6℃ | 23.2℃ | 18.6℃ | 18.5℃ | 17.4℃ | 15.4℃ |\n| 降水 | 无降水 | 无降水 | 0.1mm | 0.3mm | 0.4mm | 无降水 | 2.4mm | 2.7mm |\n| 风速 | 3m/s | 2.3m/s | 2.6m/s | 3.3m/s | 1.8m/s | 2.3m/s | 1.7m/s | 0.1m/s |\n| 风向 | 东北风 | 东南风 | 东南风 | 东北风 | 东北风 | 东北风 | 东北风 | 东北风 |\n| 气压 | 1009.9hPa | 1009.8hPa | 1007.8hPa | 1006.7hPa | 1007.8hPa | 1007.9hPa | 1007.7hPa | 1006.9hPa |\n| 湿度 | 98.6% | 75.8% | 68.6% | 67.7% | 59% | 86.4% | 96.9% | 94.7% |\n| 云量 | 10% | 71% | 72.2% | 88.7% | 95.7% | 95.2% | 99.2% | 99.7% | [...] |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 

今天时间是2026年3月25日, 上面的回答中输出的时间存在问题, 不过这个无伤大雅，可以正常查询天气

## 作业2

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [46]:
# 模型训练数据集
block_size = 32
batch_size = 32
max_iter = 5000
learn_rate = 1e-3
n_embd = 768
eval_interval = 500
eval_iters = 200

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 处理文本

In [36]:
with open('xyj.txt') as f:
        text = f.read()

# 字典、编码器(函数)、解码器(函数)
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}  #str_to_index
itos = {i:ch for i,ch in enumerate(chars)}  #index_to_str

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 文本转换token index
data = torch.tensor(encode(text), dtype=torch.long)

# 拆分数据集
n = int(len(data) * .9)
train_data = data[:n]
val_data = data[n:]

In [37]:
train_data

tensor([3833,    1,    1,  ...,  687, 2183,   16])

In [38]:
val_data

tensor([1541, 1812,   28,  ...,    1,    1,    1])

In [47]:
class Head(nn.Module):
    """单头 self-attention """
    def __init__(self, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, input_x):
        B, T, C = input_x.shape

        k = self.key(input_x)
        q = self.query(input_x)
        v = self.value(input_x)

        wei = q @ k.transpose(-2,-1) * C ** -0.5
        T = wei.shape[-1]
        tril = torch.tril(torch.ones(T,T, device=device))
        wei = wei.masked_fill(tril == 0, float('-inf'))
        wei = wei.softmax(dim=-1)

        out = wei @ v
        return out

In [48]:

class BingramLanguageModel(nn.Module):
    
    def __init__(self, block_size, vocab_size, n_embd):
        super().__init__()
        # 每个token都直接从Embedding中查询对应的logits值 以进行下一个token的推理
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # 位置编码
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # one head self-attention
        self.sa_head = Head(n_embd)
        # larg model forward
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B,T = idx.shape
        
        # idx值和targets值都是整型张量 (B,T)
        tok_emb = self.token_embedding_table(idx)  # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx指当前语料集(B,T)中的索引
        for _ in range(max_new_tokens):
            # 限定索引列的取值范围
            idx_cond = idx[:, -block_size:]
            # 推理
            logits, loss = self(idx_cond)
            # 只提取最后一个时间步的结果
            logits = logits[:, -1, :]  # (B,C)
            # 通过softmax转换为概率值
            probs = F.softmax(logits, dim=-1)  # (B,C)
            # 随机采样
            idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
            # 把采样的索引追加在当前解码序列末尾
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

In [49]:
def get_batch(train=True):
    # 选择训练或验证数据集
    data = train_data if train else val_data

    # 动态从数据集中选择位置索引
    ix = torch.randint(len(data) - block_size, (batch_size,)) # [0,103846]随机生成位置索引，向后截取block_size字符训练
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x.to(device),y.to(device)

## 模型训练


In [50]:
model = BingramLanguageModel(block_size, vocab_size, n_embd)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learn_rate)


In [51]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [52]:
def train_one_epoch():
    for iter in range(max_iter):
        if iter != 0 and iter % eval_interval == 0:
            losses = estimate_loss()
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # 批次样本
        xb, yb = get_batch('train')

        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

In [53]:
train_one_epoch()

step 500: train loss 4.4491, val loss 4.4657
step 1000: train loss 4.2482, val loss 4.2555
step 1500: train loss 4.1836, val loss 4.1943
step 2000: train loss 4.1466, val loss 4.1395
step 2500: train loss 4.1402, val loss 4.1279
step 3000: train loss 4.0878, val loss 4.0877
step 3500: train loss 4.0838, val loss 4.0795
step 4000: train loss 4.0663, val loss 4.0650
step 4500: train loss 4.0560, val loss 4.0679


In [54]:
train_one_epoch()


step 500: train loss 4.0456, val loss 4.0496
step 1000: train loss 4.0516, val loss 4.0284
step 1500: train loss 4.0389, val loss 4.0191
step 2000: train loss 4.0055, val loss 4.0187
step 2500: train loss 4.0242, val loss 4.0014
step 3000: train loss 4.0170, val loss 4.0163
step 3500: train loss 4.0189, val loss 4.0064
step 4000: train loss 3.9907, val loss 3.9997
step 4500: train loss 3.9944, val loss 3.9751


In [56]:
# 模型生成
idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(model.generate(idx, max_new_tokens=500)[0].tolist())) 

	

   妖怪的人俱是色晚，十分儿急看那祸来！”约莫能替死，老忙奏大圣祠，被他罢！山妻官执一直前架彩凤法冤有打，睡下来，定身上坐。”伯、八戒道：“正叫唤弟追着他耳上面锦？”又见光些妖精，就敢说间面有进去，听下。香茶，朝有诗易，白间有诗曰：“二仙遂而入里，腰肉在底士无任。就好善。又使糊糊糊糊的们听。”那老幼年前遇土地，退角大惊骇是他看，八戒看，扭腰我问道：“昔年无礼不说是本事出是神径至此物，刀，口诀记号喧哗，见。他？”大厦高耸，又捉他那洞中擒拿出所以相会聚贤，地，先灵神。他出长耳而逃，黄风起来，且叫他在廊之日先报仇有话。十八戒。不通透着担巡山儿就将晚，如之界，极高姓陈！”风宿星飞魄隐招堂，尚将来寻。”那大圣却如缥缈天晚，立志。
       却说那宫五庄观音乐我们那童，定善。”满面方丈人，你往弥景物觉见：
  菩萨已行者复浑家里去同去怎么就做好耍耍。”猴，“大王道：“这一根主道：“妖魔道：“骂道：“佛。”小妖魔。”遂将潜踪迹。
     莫想必然接。他不要撞去，那熊园内山魔王即便，瑞气壮斗经过花亭阁仙没事，不曾拜请他，让他扎，各饮铜兽，一顶得活活相迎之身上前进来。菩萨时过去罢。调洞坐到后


In [59]:
torch.tensor([[1, 2, 3], [4, 5, 6]])

tensor([[1, 2, 3],
        [4, 5, 6]])

In [66]:
# 从 Python 列表创建
torch.tensor([1, 2, 3])  # ✅ 正确用法

# 从 tensor 列表创建
t1 = torch.tensor([1, 2, 3])
t2 = torch.tensor([4, 5, 6])
torch.cat([t1, t2], dim=-1)  # ✅ 正确用法


tensor([1, 2, 3, 4, 5, 6])